
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 강의 - AI 에이전트 평가의 도전

## 개요

전통적인 소프트웨어 테스트 접근법은 비결정론적 특성, 창발적 행동, 맥락 의존적 반응 때문에 AI 에이전트를 평가하기에 근본적으로 불충분합니다. 이 강의에서는 왜 기존의 테스트 방식이 AI 에이전트에는 통하지 않는지 살펴보고, 특화된 평가 프레임워크를 요구하는 고유한 과제들을 알아봅니다.

AI 에이전트가 전통적인 테스트 패러다임을 어떻게 깨뜨리는지, 다단계 추론 평가의 복잡성을 이해하고, 평가가 일회성 검증 단계가 아니라 지속적인 과정으로 다뤄져야 하는 이유를 배울 것입니다.

**학습 목표**

이 강의를 마치면 다음을 할 수 있게 됩니다:
- 전통적인 소프트웨어 테스트 접근법이 AI 에이전트에 대해 불충분한 이유를 설명할 수 있다
- AI 에이전트 평가의 고유한 과제(비결정론, 창발적 행동, 맥락 의존성)를 식별할 수 있다
- 평가가 왜 지속적인 과정으로 다뤄져야 하는지 이해할 수 있다
- 적절한 평가 데이터셋 설계의 중요성을 인식할 수 있다
- 체계적인 에이전트 평가를 위한 운영 환경 구축 요구사항을 설명할 수 있다

## A. 전통적인 테스트 방식의 한계

### A1. 결정론적 테스트 패러다임

전통적인 소프트웨어 테스트는 결정론적인 입력값과 기대 출력값에 의존합니다. 개발자는 동일한 입력이 주어졌을 때 매번 완전히 동일한 결과가 나오는지를 검증하기 위해 유닛 테스트, 통합 테스트, 엔드투엔드(E2E) 테스트를 작성합니다. 이러한 방식은 동작을 예측할 수 있고 재현 가능한 기존의 소프트웨어 시스템에서는 매우 효과적입니다.

**전통적인 테스트 가정들:**
- 동일한 입력은 항상 동일한 출력을 생성함
- 동작이 명시적으로 프로그래밍되어 있으며 예측 가능함
- 정확한 문자열 일치나 숫자 비교를 통해 성공 여부를 측정할 수 있음
- 예외적인 상황(Edge case)을 예측하고 체계적으로 테스트할 수 있음

### A2. AI 에이전트가 패러다임을 깨는 방식

**AI 에이전트는 이 패러다임을 근본적으로 깨뜨립니다:**

- **비결정론성(Non-determinism)**: LLM은 템퍼러처(Temperature)와 샘플링을 통해 무작위성을 도입하므로, 동일한 입력에 대해서도 서로 다른 출력을 생성할 수 있습니다.
- **창발적 행동(Emergent behavior)**: 에이전트는 명시적으로 프로그래밍되지 않은 도구 사용 및 추론 경로에 대해 자율적인 결정을 내립니다.
- **문맥 의존성(Context dependency)**: 에이전트의 답변은 검색된 문서, 대화 기록, 외부 데이터 소스에 따라 달라집니다.
- **정성적 평가(Qualitative assessment)**: 성공 여부를 판가름하기 위해서는 정확한 문자열 일치보다는 유용성, 어조, 적절성 등에 대한 주관적인 판단이 필요한 경우가 많습니다.

간단한 예로, "샌프란시스코 날씨 어때?"라는 질문에 답하는 에이전트는 다음과 같이 반응할 수 있습니다.
- "현재 샌프란시스코는 65°F이며 맑습니다."
- "샌프란시스코 날씨: 65도, 맑은 하늘."
- "SF의 기온은 65°F이며 구름이 없습니다."

세 가지 답변 모두 정확하고 도움이 되며 적절하지만, 어느 것도 완전히 일치하지는 않습니다. 전통적인 단언(assertion) 기반 테스트(`assert output == "expected_response"`)는 세 가지 모두에서 실패할 것입니다.

## B. 에이전트 평가의 과제

### B1. 다차원 복잡성

![the-agent-evaluation-challenge.png](../Includes/images/Evaluation with MLflow/the-agent-evaluation-challenge.png)

AI 에이전트를 평가하는 것은 기존 소프트웨어보다 더 까다롭고 독특한 복잡성을 수반합니다.

**다단계 추론**: 에이전트는 여러 도구를 호출하고, 다양한 문서를 검색하며, 복잡한 추론 과정을 구축할 수 있습니다. 따라서 평가는 최종 답변뿐만 아니라 중간 과정의 품질까지 함께 측정해야 합니다.

**도구 호출 정확성**: 에이전트가 적절한 도구를 선택했는가? 올바른 매개변수(파라미터)를 전달했는가? 도구의 실행 결과를 정확하게 해석했는가? 등을 평가합니다.

**검색 품질**: 검색 증강 생성(RAG) 기반 에이전트의 경우, 검색된 문서에 유용한 정보가 포함되어 있는지, 그리고 에이전트가 여러 소스의 정보를 올바르게 종합하는지 검증해야 합니다.

### B2. 안전성 및 현실 세계의 가변성

**안전성 및 정렬(Alignment)**: 에이전트는 유해한 출력을 피하고, 사용자의 제한 사항을 준수하며, 부적절한 요청을 거부해야 합니다. 이러한 특성은 단순한 통과/실패 테스트를 넘어선 정교한 평가를 필요로 합니다.

**현실 세계의 가변성**: 실제 운영 환경의 에이전트는 다양한 사용자 질문, 예상치 못한 표현, 개발 단계에서 예측하기 어려운 예외 상황(Edge case)에 직면하게 됩니다.

이러한 과제들로 인해, AI 에이전트 특유의 확률적이고 맥락 중심적인 통계적 성격에 맞추어 설계된 더욱 정교한 평가 프레임워크가 필요합니다.

## C. 지속적인 프로세스로서의 평가

### C1. 지속적 평가 사이클

![evaluation-as-continuous-process.png](../Includes/images/Evaluation with MLflow/evaluation-as-continuous-process.png "evaluation-as-continuous-process.png")

포괄적인 테스트 제품군(test suites)을 통해 안정적인 품질 신호를 얻을 수 있는 기존 소프트웨어와 달리, AI 에이전트 평가는 다음과 같이 지속적으로 이루어지는 과정입니다.

**개발 단계**: 신속한 반복 개발 과정에서는 변경 사항이 기존 기능의 저하(regression) 없이 품질을 향상시키는지 확인하기 위해 빈번한 평가가 필요합니다.

**사전 배포 검증**: 다양한 테스트 케이스에 걸친 포괄적인 평가를 통해 에이전트가 실제 서비스에 배포되기 전 품질 기준을 충족하는지 확인합니다.

**프로덕션 모니터링**: 실제 운영 환경에서의 상호작용을 지속적으로 평가하여 품질 저하, 새롭게 나타나는 실패 패턴, 개선 기회를 포착합니다.

### C2. 지속적인 평가가 중요한 이유

이러한 지속적인 평가 사이클은 평가 인프라가 확장 가능하고 자동화되어야 하며, 개발 워크플로우에 통합되어야 함을 의미합니다. 평가 프레임워크는 다음 사항들을 지원해야 합니다.

- 개발 중 **신속한 피드백 루프**
- 배포 전 **포괄적인 검증**
- 운영 중 **지속적인 모니터링**
- 사용 패턴 변화에 따른 **데이터셋 고도화**

## D. 평가 준비

### D1. 준비가 중요한 이유

평가 프레임워크를 도입하기 전, 평가의 목적을 명확히 하고 결과를 재현할 수 있도록 만드는 것이 중요합니다. AI 에이전트는 비결정론적이며 컨텍스트에 따라 결과가 달라지므로, 기존의 단언문(Assertion) 방식의 테스트로는 한계가 있습니다. 대신 효과적인 평가를 위해서는 품질 측정 기준을 정의하고, 대표성 있는 데이터셋을 구축하며, 추적(Tracing) 기능을 활성화해야 합니다. 이를 통해 평가 주체(Judge)는 최종 답변뿐만 아니라 그 답변이 도출된 과정까지 함께 심사할 수 있습니다.

사전에 목표를 명확히 하고, 데이터셋을 선별하며, 적절한 평가 주체를 선택하면 지표가 실제 사용자 요구사항을 반영하는 선순환 구조(Feedback Loop)를 만들 수 있습니다. 또한 이를 통해 단순한 패스/실패 점수를 넘어, 추적 기록과 근거(Rationale)를 바탕으로 실패 원인을 명확히 진단할 수 있게 됩니다.

### D2. 평가 데이터셋 설계

![designing-evaluation-datasets.png](../Includes/images/Evaluation with MLflow/designing-evaluation-datasets.png "designing-evaluation-datasets.png")

평가 데이터셋은 테스트할 대상과 평가 지표의 신뢰도를 결정합니다. 데이터셋에는 최소한 입력값(질문)이 포함되어야 하며, 필요한 경우 예상 답변, 행별 가이드라인, 정보 검색(Retrieval) 및 툴 사용 현황을 반영하는 메타데이터가 함께 포함되어야 합니다.

**핵심 원칙:**

- **대표성:** 실제 운영 환경에서도 오프라인 테스트와 유사한 결과를 얻을 수 있도록, 자주 발생하고 영향력이 큰 사용자 쿼리를 포함해야 합니다.
- **예외 상황(Edge cases):** 시스템의 실패 모드를 조기에 발견할 수 있도록 모호하거나 범위에서 벗어난 프롬프트, 악의적인 공격성(Adversarial) 프롬프트를 추가해야 합니다.
- **다양성:** 추론과 정보 검색 과정의 취약점을 찾아낼 수 있도록 텍스트의 길이, 복잡성, 도메인, 사용자 숙련도를 다양하게 구성해야 합니다.
- **정답(Ground truth) 및 가이드라인:** 객관적인 질문에는 예상 답변이나 사실 관계 세트를 활용하고, 스타일·정책·완결성이 중요한 경우에는 자연어 형태의 가이드라인을 사용해야 합니다.
- **저장 및 버전 관리:** 에이전트의 발전에 발맞추어 데이터셋도 함께 업데이트될 수 있도록 JSON 파일이나 데이터프레임(가급적 Delta/Unity Catalog) 형태로 저장해야 합니다.

### D3. 운영 환경 설정 요구사항

평가 결과를 비교하고 감사(Audit)할 수 있도록 일관된 프레임워크를 구축해야 합니다.

- **MLflow 실험 및 실행(Runs):** 고정된 실험 이름을 사용하고, 각 실행에 에이전트 버전, 데이터셋 버전, 파라미터 등의 태그를 지정해야 합니다. UI를 통해 지표를 비교하고 예시별 결과물(Artifacts)을 점검할 수 있어야 합니다.
- **Unity Catalog 통합:** 액세스 제어, 버전 관리, 데이터 계보(Lineage) 기능을 활용해 데이터셋과 추적 기록을 관리해야 합니다. 엔드투엔드 추적이 가능하도록 에이전트와 종속성을 등록해야 합니다.
- **프로덕션 피드백 루프(사전 계획):** 배포 후 AI Gateway 추론 테이블을 활성화하여 요청, 응답, 추적 기록을 로깅해야 합니다. 이를 통해 모니터링을 수행하고 새로운 평가 예시 데이터를 수집할 수 있습니다.

## E. 핵심 요약

인공지능(AI) 에이전트는 비결정론적이고 창발적이며 컨텍스트(맥락)에 의존하는 특성을 지니기 때문에, 기존의 전통적인 소프트웨어 테스트 방식으로는 근본적으로 검증하기 어렵습니다. 효과적인 에이전트 평가를 위해서는 다음과 같은 요소가 필수적입니다.

1. **고유한 과제 인식**: 비결정성, 창발적 행동, 컨텍스트 의존성을 다루기 위해서는 이에 특화된 평가 접근 방식이 필요합니다.
2. **지속적인 평가 마인드셋**: 평가는 일회성 검증 단계가 아니라 제품 수명 주기 전반에 걸쳐 지속되는 과정입니다.
3. **철저한 사전 준비**: 평가의 성공 여부는 신중한 데이터셋 설계, 명확한 품질 정의, 그리고 견고한 운영 환경 구축에 달려 있습니다.
4. **체계적 접근법**: 평가 인프라는 확장 가능하고 자동화되어 있으며 개발 워크플로에 유기적으로 통합되어야 합니다

이러한 과제들을 깊이 이해하는 것이 효과적인 에이전트 평가를 구현하기 위한 기반이 됩니다. 이어지는 강의에서는 이러한 문제들을 체계적으로 해결할 수 있는 도구와 기술에 대해 자세히 살펴보겠습니다.

## ⚠️ 데모 체크포인트
<div style="border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;"></span>
    <div>
      <strong style="color: #e65100; font-size: 1.1em;">데모 체크포인트</strong>
      <p style="margin: 8px 0 0 0; color: #333;">첫 번째 데모 &lt;strong>01 Demo - Agent Setup&lt;/strong>으로 이동해 이번 교육 전반에서 사용할 에이전트와 UC 자산 설정을 시작하세요. 완료 후 이 강의 노트북으로 돌아와 학습을 이어가세요.</p>
    </div>
  </div>
</div>

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>